# TP Jour 4 – UC-08 : Analyse des données, KPI et visualisation
**Retail Intelligence Platform – IPSSI**

- **Groupe :**
- **Date :**
- **Parcours UC-07 retenu :** B (local / open source) – MinIO + Apache Iceberg + Spark + Trino

### Objectifs de l'analyse

Question métier issue du Jour 1 : **comment se répartit et évolue le chiffre d'affaires des ventes magasin ?**

1. Mesurer l'activité des ventes magasin (chiffre d'affaires net, quantités, tickets, panier moyen) sur le lot initial TPCx-BB.
2. Observer l'évolution mensuelle des ventes (2001-2005) et comparer les 12 magasins.
3. Identifier les catégories de produits qui contribuent le plus au chiffre d'affaires.
4. Mesurer le volume de retours magasin en précisant ses limites.

Les données TPCx-BB sont **synthétiques** : les résultats illustrent une démarche analytique et ne décrivent pas une entreprise réelle.

## 0. Environnement de référence

| Composant | Version / rôle | Où |
| --- | --- | --- |
| Python | 3.12 | conteneur `lake` (`/opt/venv`, géré par uv) |
| Apache Spark / PySpark | 4.1.4, mode `local[*]` | conteneur `lake` |
| Apache Iceberg | 1.12.0 (runtime Spark), catalogue REST 1.10.1 | `iceberg-rest`, catalogue Spark `iceberg` |
| MinIO | RELEASE.2026-08-04 (stockage objet S3, bucket `warehouse`) | `minio` |
| Trino | 484 (requêtes SQL, catalogue `iceberg`) | `trino` |
| Pandas / Matplotlib | voir cellule suivante | conteneur `lake` |

Démarrage : `docker compose up -d` puis Jupyter `http://localhost:8888/?token=tp`.
**Prérequis :** le notebook `TP4_00_chargement_lakehouse.ipynb` doit avoir été exécuté (il remplit les couches Bronze / Silver / Gold).

In [1]:
import platform
import subprocess
import sys

import matplotlib
import pandas as pd
import pyspark
import trino

java = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()[0]
print("OS          :", platform.platform())
print("Python      :", sys.version.split()[0], sys.executable)
print("Java        :", java)
print("PySpark     :", pyspark.__version__)
print("Pandas      :", pd.__version__)
print("Matplotlib  :", matplotlib.__version__)
print("Trino client:", trino.__version__ if hasattr(trino, "__version__") else "installé")

OS          : Linux-7.0.0-43-generic-x86_64-with-glibc2.41
Python      : 3.12.16 /opt/venv/bin/python
Java        : openjdk version "17.0.21" 2026-10-20
PySpark     : 4.1.4
Pandas      : 2.3.3
Matplotlib  : 3.11.2
Trino client: 0.340.0


In [ ]:
conn = trino.dbapi.connect(host="trino", port=8080, user="etudiant", catalog="iceberg")
cur = conn.cursor()
cur.execute("SELECT node_version FROM system.runtime.nodes")
print("Trino       :", cur.fetchone()[0])
cur.execute("SHOW SCHEMAS FROM iceberg")
print("Schémas Iceberg vus par Trino :", sorted(r[0] for r in cur.fetchall() if r[0] != "information_schema"))

Trino       : 484
Schémas Iceberg vus par Trino : ['bronze', 'gold', 'silver', 'system']


## 1. Import des bibliothèques

In [3]:
from pathlib import Path

import matplotlib.pyplot as plt
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

## 2. Création de la session Spark

Le catalogue Iceberg (`iceberg`), son URI REST et l'accès à MinIO sont déclarés dans `docker/spark/spark-defaults.conf` : la session n'a pas besoin de les répéter.

In [4]:
spark = (
    SparkSession.builder
    .appName("TP4-Retail-Intelligence")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Version Spark :", spark.version)
print("Master        :", spark.sparkContext.master)
print("Catalogue     :", spark.conf.get("spark.sql.catalog.iceberg.uri"), "->", spark.conf.get("spark.sql.catalog.iceberg.warehouse"))
for ns in ["bronze", "silver", "gold"]:
    print(f"iceberg.{ns:7s}:", ", ".join(sorted(r.tableName for r in spark.sql(f"SHOW TABLES IN iceberg.{ns}").collect())))

26/11/19 14:11:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Version Spark : 4.1.4
Master        : local[*]


Catalogue     : http://iceberg-rest:8181 -> s3://warehouse/


iceberg.bronze : customer, customer_address, customer_demographics, date_dim, household_demographics, income_band, inventory, item, item_marketprices, product_reviews, promotion, reason, ship_mode, store, store_returns, store_sales, time_dim, warehouse, web_clickstreams, web_page, web_returns, web_sales, web_site
iceberg.silver : date_dim, item, store, store_returns, store_sales
iceberg.gold   : retours_magasin_mensuels, ventes_magasin_mensuelles, ventes_magasin_par_produit


## 3. Chargement des jeux de données

Les jeux viennent du lakehouse Iceberg (MinIO), produits par `TP4_00_chargement_lakehouse.ipynb` à partir des fichiers TPCx-BB du lot initial, avec les règles de nettoyage du Jour 3 (UC-06).

### 3.1 Jeux Silver

| Table | Rôle dans l'analyse |
| --- | --- |
| `iceberg.silver.store_sales` | fait : lignes de ticket des ventes magasin |
| `iceberg.silver.store_returns` | fait : lignes de retour magasin |
| `iceberg.silver.item` | dimension produit (catégorie, classe) |
| `iceberg.silver.store` | dimension magasin |
| `iceberg.silver.date_dim` | dimension calendrier |

In [5]:
df_ventes = spark.table("iceberg.silver.store_sales")
df_retours = spark.table("iceberg.silver.store_returns")
df_produits = spark.table("iceberg.silver.item")
df_magasins = spark.table("iceberg.silver.store")
df_dates = spark.table("iceberg.silver.date_dim")

df_ventes.select("ss_sold_date_sk", "ss_store_sk", "ss_ticket_number", "ss_item_sk", "ss_quantity", "ss_net_paid", "ss_net_profit").show(5)

+---------------+-----------+----------------+----------+-----------+-----------+-------------+
|ss_sold_date_sk|ss_store_sk|ss_ticket_number|ss_item_sk|ss_quantity|ss_net_paid|ss_net_profit|
+---------------+-----------+----------------+----------+-----------+-----------+-------------+
|          36890|         10|               0|     10000|         54|    4500.36|      -856.98|
|          36890|          4|               1|     10413|         84|      31.60|      -290.96|
|          36890|         11|               2|      3963|         94|    2092.44|     -3600.20|
|          36890|          5|               5|      8633|         95|    6840.00|      2421.55|
|          36890|         11|               6|     15336|         52|    2692.56|       902.20|
+---------------+-----------+----------------+----------+-----------+-----------+-------------+
only showing top 5 rows


### 3.2 Jeux Gold

| Table | Granularité |
| --- | --- |
| `iceberg.gold.ventes_magasin_mensuelles` | un magasin × un mois |
| `iceberg.gold.ventes_magasin_par_produit` | un article |
| `iceberg.gold.retours_magasin_mensuels` | un magasin × un mois (date de retour) |

In [6]:
gold_mensuel = spark.table("iceberg.gold.ventes_magasin_mensuelles")
gold_produit = spark.table("iceberg.gold.ventes_magasin_par_produit")
gold_retours = spark.table("iceberg.gold.retours_magasin_mensuels")

gold_mensuel.orderBy("mois", "s_store_sk").show(5)
gold_produit.orderBy(F.desc("ca_net")).show(5)

+-------+----------+----------------+------------+-------+---------+----------+--------+----------+----------+
|   mois|s_store_sk|      s_store_id|s_store_name|s_state|nb_lignes|nb_tickets|quantite|    ca_net|profit_net|
+-------+----------+----------------+------------+-------+---------+----------+--------+----------+----------+
|2001-01|         0|AAAAAAAAAAAAAAAA|      Forest|     CO|      861|       111|   44609|1386332.96|-846846.06|
|2001-01|         1|AAAAAAAAAAAAAAAB|    Hillside|     TN|      932|       129|   47121|1564162.75|-754801.44|
|2001-01|         2|AAAAAAAAAAAAAAAC|    Sherwood|     CA|      810|       113|   42102|1471262.07|-647082.55|
|2001-01|         3|AAAAAAAAAAAAAAAD|     Pioneer|     TX|      987|       133|   49402|1657308.51|-736476.39|
|2001-01|         4|AAAAAAAAAAAAAAAE|     Elmwood|     KY|      950|       133|   48855|1640426.13|-794777.25|
+-------+----------+----------------+------------+-------+---------+----------+--------+----------+----------+
o

+---------+----------------+--------------------+--------------+--------------------+---------+--------+-----------+-----------+-----------+
|i_item_sk|       i_item_id|      i_product_name|    i_category|             i_class|nb_lignes|quantite|     ca_net| ca_net_ttc| profit_net|
+---------+----------------+--------------------+--------------+--------------------+---------+--------+-----------+-----------+-----------+
|    10000|AAAAAAAAAAAAAOUQ|vwPR6l3UPyCIv5blj...|Home & Kitchen|Kitchen Utensils ...|     9029|  454954|15408393.55|16108841.77|-7521122.26|
|    10003|AAAAAAAAAAAAAOUT|                Z2kc|   Electronics|Computer Accessories|     8761|  439086|14933870.10|15614283.97|-7411910.29|
|    10001|AAAAAAAAAAAAAOUR|     6VxbaI6thqY49zO|         Music|                 R&B|     8269|  420138|14572124.66|15240549.34|-6753145.81|
|    10002|AAAAAAAAAAAAAOUS|bwTLbeGP8aT6bW7Yj...|         Music|    Alternative Rock|     7505|  377027|12817437.86|13397798.56|-6147236.12|
|    17502|AA

## 4. Vérification des schémas et des volumes

Pour chaque jeu : nom exact, provenance, format, schéma, granularité, contrôles qualité disponibles, pertinence pour l'analyse.

In [7]:
jeux = {
    "silver.store_sales": df_ventes,
    "silver.store_returns": df_retours,
    "silver.item": df_produits,
    "silver.store": df_magasins,
    "silver.date_dim": df_dates,
    "gold.ventes_magasin_mensuelles": gold_mensuel,
    "gold.ventes_magasin_par_produit": gold_produit,
    "gold.retours_magasin_mensuels": gold_retours,
}
volumes = pd.DataFrame(
    [(nom, df.count(), len(df.columns)) for nom, df in jeux.items()],
    columns=["jeu", "lignes", "colonnes"],
)
volumes

,jeu,lignes,colonnes
0,silver.store_sales,667579,25
1,silver.store_returns,37902,22
2,silver.item,17820,24
3,silver.store,12,31
4,silver.date_dim,109573,30
5,gold.ventes_magasin_mensuelles,720,10
6,gold.ventes_magasin_par_produit,17820,10
7,gold.retours_magasin_mensuels,948,5


In [8]:
for nom in ["silver.store_sales", "silver.store_returns", "gold.ventes_magasin_mensuelles", "gold.ventes_magasin_par_produit"]:
    print("=====", nom)
    jeux[nom].printSchema()

===== silver.store_sales
root
 |-- ss_sold_date_sk: long (nullable = true)
 |-- ss_sold_time_sk: long (nullable = true)
 |-- ss_item_sk: long (nullable = true)
 |-- ss_customer_sk: long (nullable = true)
 |-- ss_cdemo_sk: long (nullable = true)
 |-- ss_hdemo_sk: long (nullable = true)
 |-- ss_addr_sk: long (nullable = true)
 |-- ss_store_sk: long (nullable = true)
 |-- ss_promo_sk: long (nullable = true)
 |-- ss_ticket_number: long (nullable = true)
 |-- ss_quantity: long (nullable = true)
 |-- ss_wholesale_cost: decimal(15,2) (nullable = true)
 |-- ss_list_price: decimal(15,2) (nullable = true)
 |-- ss_sales_price: decimal(15,2) (nullable = true)
 |-- ss_ext_discount_amt: decimal(15,2) (nullable = true)
 |-- ss_ext_sales_price: decimal(15,2) (nullable = true)
 |-- ss_ext_wholesale_cost: decimal(15,2) (nullable = true)
 |-- ss_ext_list_price: decimal(15,2) (nullable = true)
 |-- ss_ext_tax: decimal(15,2) (nullable = true)
 |-- ss_coupon_amt: decimal(15,2) (nullable = true)
 |-- ss_net_

In [ ]:
granularite = df_ventes.agg(
    F.count("*").alias("lignes"),
    F.countDistinct("ss_ticket_number").alias("tickets_distincts"),
    F.countDistinct("ss_store_sk", "ss_ticket_number").alias("couples_magasin_ticket"),
    F.countDistinct("ss_ticket_number", "ss_item_sk").alias("couples_ticket_article"),
    F.countDistinct("ss_customer_sk").alias("clients_distincts"),
).toPandas()
granularite["lignes_par_ticket"] = (granularite["lignes"] / granularite["tickets_distincts"]).round(2)
granularite

,lignes,tickets_distincts,couples_magasin_ticket,couples_ticket_article,clients_distincts,lignes_par_ticket
0,667579,89100,89100,663443,37336,7.49


In [ ]:
periode_ventes = (
    df_ventes.join(df_dates, df_ventes.ss_sold_date_sk == df_dates.d_date_sk)
    .agg(F.min("d_date").alias("debut"), F.max("d_date").alias("fin"), F.countDistinct(F.date_format("d_date", "yyyy-MM")).alias("mois"))
    .withColumn("jeu", F.lit("store_sales"))
)
periode_retours = (
    df_retours.join(df_dates, df_retours.sr_returned_date_sk == df_dates.d_date_sk)
    .agg(F.min("d_date").alias("debut"), F.max("d_date").alias("fin"), F.countDistinct(F.date_format("d_date", "yyyy-MM")).alias("mois"))
    .withColumn("jeu", F.lit("store_returns"))
)
periode_ventes.unionByName(periode_retours).select("jeu", "debut", "fin", "mois").toPandas()

,jeu,debut,fin,mois
0,store_sales,2001-01-01,2005-12-13,60
1,store_returns,2001-01-25,2007-11-14,83


### 4.1 Inventaire des jeux utilisés (J4-01)

Provenance commune : kit TPCx-BB fourni (`retail_project/data`, lot initial, fichiers `.dat` séparés par `|`), chargé en Bronze puis nettoyé en Silver par `TP4_00_chargement_lakehouse.ipynb`. Format physique : table Apache Iceberg, fichiers Parquet dans MinIO (`s3://warehouse/`).

| Jeu de données | Zone | Format | Granularité | Lignes | Contrôles qualité | Usage prévu |
|---|---|---|---|---|---|---|
| `store_sales` | Silver | Iceberg / Parquet | une ligne d'article dans un ticket | 667 579 | clés requises non nulles, quantité > 0, doublons exacts, intégrité vers item / store / date_dim | KPI, statistiques |
| `store_returns` | Silver | Iceberg / Parquet | une ligne d'article retourné | 37 902 | clés requises non nulles, quantité retournée > 0, doublons exacts | KPI retours |
| `item` | Silver | Iceberg / Parquet | un article | 17 820 | `i_item_sk` unique et non nul | dimension produit |
| `store` | Silver | Iceberg / Parquet | un magasin | 12 | `s_store_sk` unique | dimension magasin |
| `date_dim` | Silver | Iceberg / Parquet | un jour | 109 573 | `d_date_sk` unique, `d_date` non nul | dimension calendrier |
| `ventes_magasin_mensuelles` | Gold | Iceberg / Parquet | un magasin × un mois | 720 | Σ lignes et Σ CA = Silver | analyse temporelle, comparaison magasins |
| `ventes_magasin_par_produit` | Gold | Iceberg / Parquet | un article | 17 820 | Σ lignes et Σ CA = Silver | analyse produits / catégories |
| `retours_magasin_mensuels` | Gold | Iceberg / Parquet | un magasin × un mois de retour | 948 | — | volume de retours |

Pertinence : ces jeux sont ceux de la question métier du Jour 1 (CA des ventes magasin) et disposent d'une date exploitable, d'un magasin et d'une catégorie produit vérifiés par jointure.

### 4.2 Qualité : valeurs manquantes, doublons, anomalies

In [11]:
def valeurs_manquantes(df, colonnes):
    total = df.count()
    ligne = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in colonnes]).first()
    return pd.DataFrame(
        [(c, ligne[c], round(100 * ligne[c] / total, 2)) for c in colonnes],
        columns=["colonne", "nulls", "pct"],
    )

valeurs_manquantes(df_ventes, [
    "ss_sold_date_sk", "ss_store_sk", "ss_ticket_number", "ss_item_sk", "ss_customer_sk",
    "ss_promo_sk", "ss_hdemo_sk", "ss_addr_sk", "ss_quantity", "ss_net_paid", "ss_net_profit",
])

,colonne,nulls,pct
0,ss_sold_date_sk,0,0.00
1,ss_store_sk,0,0.00
2,ss_ticket_number,0,0.00
3,ss_item_sk,0,0.00
4,ss_customer_sk,0,0.00
5,ss_promo_sk,158,0.02
6,ss_hdemo_sk,295,0.04
7,ss_addr_sk,448,0.07
8,ss_quantity,0,0.00
9,ss_net_paid,0,0.00


In [ ]:
nb_doublons = df_ventes.count() - df_ventes.dropDuplicates().count()
anomalies = df_ventes.agg(
    F.lit(nb_doublons).alias("doublons_exacts"),
    F.sum((F.col("ss_quantity") <= 0).cast("int")).alias("quantite_non_positive"),
    F.sum((F.col("ss_net_paid") < 0).cast("int")).alias("ca_net_negatif"),
    F.sum((F.col("ss_net_paid") == 0).cast("int")).alias("ca_net_nul"),
    F.sum((F.col("ss_net_profit") < 0).cast("int")).alias("profit_negatif"),
).toPandas()
anomalies["pct_profit_negatif"] = round(100 * anomalies["profit_negatif"] / df_ventes.count(), 1)
anomalies

,doublons_exacts,quantite_non_positive,ca_net_negatif,ca_net_nul,profit_negatif,pct_profit_negatif
0,0,0,0,3954,495971,74.3


In [13]:
orphelins = pd.DataFrame([
    ("store_sales -> item", df_ventes.join(df_produits, df_ventes.ss_item_sk == df_produits.i_item_sk, "left_anti").count()),
    ("store_sales -> store", df_ventes.join(df_magasins, df_ventes.ss_store_sk == df_magasins.s_store_sk, "left_anti").count()),
    ("store_sales -> date_dim", df_ventes.join(df_dates, df_ventes.ss_sold_date_sk == df_dates.d_date_sk, "left_anti").count()),
    ("store_returns -> store_sales (ticket)", df_retours.join(df_ventes, df_retours.sr_ticket_number == df_ventes.ss_ticket_number, "left_anti").count()),
], columns=["relation", "orphelins"])
orphelins

,relation,orphelins
0,store_sales -> item,0
1,store_sales -> store,0
2,store_sales -> date_dim,0
3,store_returns -> store_sales (ticket),0


**Constats qualité :**

- Aucune clé manquante sur les colonnes utilisées par les KPI (date, magasin, ticket, article, quantité, montant net) ; les clés secondaires (client, promotion, foyer, adresse) peuvent être vides et ne sont pas utilisées.
- Aucun doublon exact, aucune quantité nulle ou négative, aucun montant net négatif.
- 3 954 lignes (0,6 %) ont un montant net égal à 0 : elles sont conservées (quantité vendue réelle, montant nul possible après remise ou coupon) et comptent dans les quantités et les tickets.
- Toutes les ventes retrouvent leur article, leur magasin et leur date : les jointures ne perdent ni ne multiplient de lignes.
- `ss_net_profit` est négatif sur environ trois lignes sur quatre : caractéristique du générateur synthétique. Le profit n'est donc **pas** retenu comme indicateur de marge.
- Le couple (ticket, article) n'est pas unique : une jointure retours → ventes sur ce couple multiplierait des lignes. Les retours sont donc analysés séparément, par date de retour.
- Les retours couvrent 2001-01 à 2007-11 alors que les ventes s'arrêtent en 2005-12 : un taux de retour mensuel n'a de sens que sur la période commune.

## 5. Dictionnaire des KPI (J4-02)

KPI retenus (calculables et vérifiés sur les données disponibles) :

| KPI | Définition métier | Formule | Champs utilisés | Filtres | Granularité | Unité | Limites |
|---|---|---|---|---|---|---|---|
| **CA net magasin** | Montant net encaissé sur les ventes magasin, hors taxes | `SUM(ss_net_paid)` | `store_sales.ss_net_paid` | lot initial, lignes Silver | total, par mois, par magasin, par catégorie | unité monétaire du fichier (non précisée par le kit) | hors taxes ; hors ventes web ; données synthétiques |
| **Quantités vendues** | Nombre d'unités d'articles vendues | `SUM(ss_quantity)` | `store_sales.ss_quantity` | `ss_quantity > 0` | total, par mois, par magasin, par catégorie | unités | ≠ nombre de lignes, ≠ nombre de tickets |
| **Nombre de tickets** | Nombre de passages en caisse | `COUNT(DISTINCT ss_ticket_number)` | `store_sales.ss_ticket_number` | lot initial | total, par mois, par magasin | tickets | un ticket = un magasin (vérifié : 89 100 tickets = 89 100 couples magasin/ticket) |
| **Panier moyen** | CA net moyen par ticket | `SUM(ss_net_paid) / COUNT(DISTINCT ss_ticket_number)` | `ss_net_paid`, `ss_ticket_number` | lot initial | total, par magasin, par mois | unité monétaire / ticket | calculé sur les tickets, **pas** sur les lignes (≈ 7,5 lignes par ticket) |
| **Volume de retours** | Unités et montant retournés en magasin | `SUM(sr_return_quantity)`, `SUM(sr_return_amt)` | `store_returns` | date de retour | par mois de retour, par magasin | unités, unité monétaire | rattaché à la date de retour, pas à la date de vente ; retours jusqu'en 2007-11 |
| **Taux de retour (quantité)** | Part des unités vendues retournées | `SUM(sr_return_quantity) / SUM(ss_quantity)` | `sr_return_quantity`, `ss_quantity` | numérateur : retours datés 2001-01 → 2005-12 ; dénominateur : ventes de la même période | période entière ou par magasin | % | numérateur et dénominateur sur des dates différentes (retour vs vente) : indicateur approché |

**KPI non retenus :**

| KPI | Raison |
|---|---|
| Marge | `ss_net_profit` est négatif sur ~74 % des lignes (génération synthétique) et les coûts ne sont pas qualifiés : la marge ne serait pas interprétable. |
| Répartition par canal | Le canal n'est pas une colonne : seule la séparation magasin (`store_sales`) / web (`web_sales`) existe. Hors périmètre de cette analyse magasin. |
| Panier moyen par client | `ss_customer_sk` est renseigné mais la fidélité des clients synthétiques n'est pas vérifiée. |

## 6. Calcul des KPI

### KPI 1 – 

### KPI 2 – 

### KPI 3 – 

### Export des KPI (`analytics/outputs/kpi.csv`)

## 7. Statistiques descriptives (J4-03)
Nombre d'enregistrements, min / max, moyenne / médiane (si pertinent), sommes des mesures additives, répartition par dimension, évolution temporelle si date exploitable.

### Export (`analytics/outputs/statistiques.csv`)

**Commentaire :**

*À compléter.*

## 8. Préparation des dossiers d'export

## 9. Analyse métier n°1 (J4-04)
- **Question métier :** 
- **Données utilisées :** 
- **Méthode de calcul :** 

**Résultats :**

**Interprétation :**

**Limites :**

## 10. Visualisation n°1 (J4-05)
Titre explicite, axes et unités, légende si nécessaire, source des données. Export PNG dans `analytics/visualizations/`.

**Source :** 

**Ce que montre le graphique / résultat principal / interprétation / limites :**

## 11. Analyse métier n°2 (J4-04)
- **Question métier :** 
- **Données utilisées :** 
- **Méthode de calcul :** 

**Résultats :**

**Interprétation :**

**Limites :**

## 12. Visualisation n°2 (J4-05)

**Source :** 

**Ce que montre le graphique / résultat principal / interprétation / limites :**

## 13. Interprétation et limites – Constats (J4-06)
Au minimum **2 constats**. Rappel : corrélation ≠ causalité ; les données TPCx-BB sont synthétiques.

### Constat 1
- **Résultat observé :** 
- **Interprétation possible :** 
- **Limite :** 
- **Recommandation :** 

### Constat 2
- **Résultat observé :** 
- **Interprétation possible :** 
- **Limite :** 
- **Recommandation :** 

## 14. Conclusion et recommandations

*À compléter.*

## 15. Instructions de reproduction

*À compléter.*

## 16. Tests UC-08

| Contrôle | État | Preuve |
|---|---|---|
| Les jeux de données utilisés sont identifiés. | ☐ | Référence |
| Les schémas et granularités ont été vérifiés. | ☐ | Capture / note |
| Les KPI disposent d'une définition explicite. | ☐ | Dictionnaire |
| Les calculs sont reproductibles. | ☐ | Script / notebook |
| Les statistiques sont cohérentes avec les données. | ☐ | Résultat |
| Les visualisations sont lisibles et légendées. | ☐ | Graphiques |
| Les interprétations sont justifiées. | ☐ | Rapport |
| Les limites des données sont mentionnées. | ☐ | Rapport |

## 17. Questions de recul

**1. Pourquoi faut-il vérifier la granularité avant de calculer un KPI ?**

*À compléter.*

**2. Quelle différence existe entre un nombre de lignes et un nombre de transactions ?**

*À compléter.*

**3. Quelles limites avez-vous rencontrées dans les données utilisées pour vos analyses ?**

*À compléter.*

**4. Quels indicateurs n'avez-vous pas pu calculer, et pour quelles raisons ?**

*À compléter.*

**5. Comment avez-vous vérifié la cohérence des résultats ?**

*À compléter.*

**6. Quelle différence existe entre un constat statistique et une conclusion causale ?**

*À compléter.*

**7. Pourquoi les données synthétiques doivent-elles être identifiées dans la restitution ?**

*À compléter.*

## 18. Partie complémentaire (facultatif) – CO-01 : analyse supplémentaire

**Données, méthode, résultats, limites :**